In [1]:
# =============
# 环境、路径与全局参数
# =============

import json
import shutil
import random
from pathlib import Path
from PIL import Image, ImageOps

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path("..").resolve()

JSON_PATH = ROOT / "train_image.json"
IMAGE_DIR = ROOT / "images"

# fire 标注框目录。目录内通常是 labelme 风格 json。
FIRE_BOX_JSON_DIR = ROOT / "images_fire"

SPLIT_DIR = ROOT / "fire_splits"
ENSEMBLE_DIR = SPLIT_DIR / "ensemble_splits"
TRAIN_META_DIR = SPLIT_DIR / "train"
TEST_META_DIR = SPLIT_DIR / "test"
MODEL_SAVE_DIR = SPLIT_DIR / "models"

USE_FIRE_OFFLINE_AUG = True
USE_NO_FIRE_OFFLINE_AUG = True

# fire 增强类型；暂时保留 hflip/vflip/hvflip，所有增强都会同步变换 bbox json
FIRE_AUG_TYPES = ["hflip", "vflip", "hvflip"]

# no_fire 用更强增强，增强灯光、反光、过曝 hard negative 多样性
NO_FIRE_AUG_TYPES = ["hflip", "vflip", "hvflip", "rotate_pos", "rotate_neg"]
USE_ALL_MANUAL_NO_FIRE = True

USE_MANUAL_NO_FIRE_CROPS = True

MANUAL_NO_FIRE_DIR = ROOT / "manual_no_fire_crops"

AUG_IMAGE_DIR = SPLIT_DIR / "augmented_images"

AUG_NO_FIRE_DIR = AUG_IMAGE_DIR / "no_fire"
AUG_FIRE_DIR = AUG_IMAGE_DIR / "fire"

ALL_VALID_CSV = SPLIT_DIR / "all_valid_images.csv"
TEST_CSV = SPLIT_DIR / "test_fixed_real_only.csv"
VAL_CSV = SPLIT_DIR / "val_fixed_real_only.csv"
TRAIN_BASE_CSV = SPLIT_DIR / "train_base_pool.csv"
ENSEMBLE_SUMMARY_CSV = ENSEMBLE_DIR / "ensemble_summary.csv"

RANDOM_SEED = 35

# 是否强制重建 fire 框外 no_fire crop。
FORCE_REBUILD_FIRE_BOX_CROPS = False

# 是否强制重新划分 train / val / test。
FORCE_RESPLIT = True

TRAIN_PER_CLASS = 500
TEST_PER_CLASS = 150
VAL_PER_CLASS = 50

MAX_MANUAL_PREVIEW = 10

N_ENSEMBLE = 1

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_META_DIR.mkdir(parents=True, exist_ok=True)
TEST_META_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_NO_FIRE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("JSON_PATH:", JSON_PATH)
print("IMAGE_DIR:", IMAGE_DIR)
print("FIRE_BOX_JSON_DIR:", FIRE_BOX_JSON_DIR)
print("SPLIT_DIR:", SPLIT_DIR)

ROOT: E:\Programming\Python\DeepLearning\比赛
JSON_PATH: E:\Programming\Python\DeepLearning\比赛\train_image.json
IMAGE_DIR: E:\Programming\Python\DeepLearning\比赛\images
FIRE_BOX_JSON_DIR: E:\Programming\Python\DeepLearning\比赛\images_fire
SPLIT_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits


In [2]:
# =============
# 环境、路径与全局参数
# =============

import json
import shutil
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()

JSON_PATH = ROOT / "train_image.json"
IMAGE_DIR = ROOT / "images"

# fire 标注框目录：里面放原始 fire 图片对应的 LabelMe json
# 例如：images_fire/20250526_firesmoke_00019.json
FIRE_BOX_JSON_DIR = ROOT / "images_fire"

SPLIT_DIR = ROOT / "fire_splits"
ENSEMBLE_DIR = SPLIT_DIR / "ensemble_splits"
TRAIN_META_DIR = SPLIT_DIR / "train"
TEST_META_DIR = SPLIT_DIR / "test"
MODEL_SAVE_DIR = SPLIT_DIR / "models"

MANUAL_NO_FIRE_DIR = ROOT / "manual_no_fire_crops"

AUG_IMAGE_DIR = SPLIT_DIR / "augmented_images"
AUG_NO_FIRE_DIR = AUG_IMAGE_DIR / "no_fire"
AUG_FIRE_DIR = AUG_IMAGE_DIR / "fire"

# 新增：增强后的 fire 标注框保存目录
AUG_BOX_JSON_DIR = SPLIT_DIR / "augmented_box_jsons"
AUG_FIRE_BOX_JSON_DIR = AUG_BOX_JSON_DIR / "fire"

ALL_VALID_CSV = SPLIT_DIR / "all_valid_images.csv"
TEST_CSV = SPLIT_DIR / "test_fixed_real_only.csv"
VAL_CSV = SPLIT_DIR / "val_fixed_real_only.csv"
TRAIN_BASE_CSV = SPLIT_DIR / "train_base_pool.csv"
ENSEMBLE_SUMMARY_CSV = ENSEMBLE_DIR / "ensemble_summary.csv"

RANDOM_SEED = 35
FORCE_RESPLIT = True

TRAIN_PER_CLASS = 500
TEST_PER_CLASS = 150
VAL_PER_CLASS = 50

N_ENSEMBLE = 1
MAX_MANUAL_PREVIEW = 10

USE_MANUAL_NO_FIRE_CROPS = True
USE_NO_FIRE_OFFLINE_AUG = True
USE_FIRE_OFFLINE_AUG = True

FIRE_AUG_TYPES = ["hflip", "vflip", "hvflip"]
NO_FIRE_AUG_TYPES = ["hflip", "vflip", "hvflip", "rotate_pos", "rotate_neg"]

# 默认 fire 增强只在 fire 原图不足 TRAIN_PER_CLASS 时补齐。
# 如果你希望即使 fire 足够，也额外生成 fire 增强图，可以把它设为 1 或 2。
# 例如 FIRE_EXTRA_AUG_PER_ORIGINAL = 1 表示每张有框 fire 原图额外增强 1 张。
FIRE_EXTRA_AUG_PER_ORIGINAL = 1

# 即使 no_fire 数量已经足够，也额外生成一部分 no_fire 离线增强图，
# 用于制造更多灯光、反光、过曝、旋转等 hard negative。
# 如果不想额外生成 no_fire 增强，把它设为 0。
NO_FIRE_FORCE_EXTRA_AUG_COUNT = 200

# fire 增强必须要求有对应标注框，否则增强后无法生成可靠 bbox
REQUIRE_FIRE_BOX_FOR_FIRE_AUG = True

# 如果原图尺寸和 LabelMe json 里的 imageWidth/imageHeight 不一致，则不用于 fire 增强
REQUIRE_ANN_SIZE_MATCH = True

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_META_DIR.mkdir(parents=True, exist_ok=True)
TEST_META_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_NO_FIRE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("JSON_PATH:", JSON_PATH)
print("IMAGE_DIR:", IMAGE_DIR)
print("FIRE_BOX_JSON_DIR:", FIRE_BOX_JSON_DIR)
print("SPLIT_DIR:", SPLIT_DIR)
print("AUG_FIRE_BOX_JSON_DIR:", AUG_FIRE_BOX_JSON_DIR)

# =============
# LabelMe 标注读取与 bbox 变换工具
# =============

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RESAMPLE = Image.Resampling.BICUBIC


def find_fire_box_json(filename):
    """
    根据图片文件名寻找对应的 LabelMe json。
    优先查 FIRE_BOX_JSON_DIR / stem.json。
    若没有，再递归查找。
    """
    filename = str(filename)
    stem = Path(filename).stem

    direct = FIRE_BOX_JSON_DIR / f"{stem}.json"
    if direct.exists():
        return direct

    if FIRE_BOX_JSON_DIR.exists():
        matches = list(FIRE_BOX_JSON_DIR.rglob(f"{stem}.json"))
        if len(matches) > 0:
            return matches[0]

    return None


def load_labelme_json(json_path):
    json_path = Path(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def count_fire_shapes_from_labelme(data):
    shapes = data.get("shapes", [])
    cnt = 0

    for s in shapes:
        label = str(s.get("label", "")).lower().strip()
        if label == "fire":
            cnt += 1

    return cnt


def get_labelme_size(data):
    w = data.get("imageWidth", None)
    h = data.get("imageHeight", None)
    return w, h


def clip_point(x, y, w, h):
    x = float(max(0, min(w, x)))
    y = float(max(0, min(h, y)))
    return [x, y]


def transform_point_for_aug(x, y, w, h, aug_type):
    """
    对 LabelMe 点坐标做与图片增强一致的变换。
    LabelMe 坐标是连续坐标，这里用 w-x / h-y。
    """
    x = float(x)
    y = float(y)

    if aug_type in ["hflip", "hvflip"]:
        x = w - x

    if aug_type in ["vflip", "hvflip"]:
        y = h - y

    return clip_point(x, y, w, h)


def rect_points_from_transformed_points(points, old_point_count=4):
    """
    将翻转后的点重新整理为矩形框。
    兼容两种 LabelMe rectangle：
    1. 两点格式：[[x1,y1],[x2,y2]]
    2. 四点格式：[[x1,y1],[x2,y1],[x2,y2],[x1,y2]]
    """
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    if old_point_count == 2:
        return [
            [x1, y1],
            [x2, y2]
        ]

    return [
        [x1, y1],
        [x2, y1],
        [x2, y2],
        [x1, y2]
    ]


def transform_shape_for_aug(shape, w, h, aug_type):
    """
    变换单个 LabelMe shape。
    对 rectangle：翻转后重新规整为矩形。
    对 polygon / 其他形状：逐点变换。
    """
    new_shape = deepcopy(shape)

    points = shape.get("points", [])
    old_point_count = len(points)

    new_points = [
        transform_point_for_aug(x, y, w, h, aug_type)
        for x, y in points
    ]

    shape_type = str(shape.get("shape_type", "")).lower().strip()

    if shape_type == "rectangle":
        new_shape["points"] = rect_points_from_transformed_points(
            new_points,
            old_point_count=old_point_count
        )
    else:
        new_shape["points"] = new_points

    return new_shape


def transform_labelme_for_aug(
        source_json_path,
        aug_type,
        image_w,
        image_h,
        aug_image_filename,
        aug_image_path
):
    """
    根据增强方式，同步生成增强后的 LabelMe json。
    """
    data = load_labelme_json(source_json_path)
    new_data = deepcopy(data)

    new_shapes = []

    for shape in data.get("shapes", []):
        new_shape = transform_shape_for_aug(
            shape=shape,
            w=image_w,
            h=image_h,
            aug_type=aug_type
        )
        new_shapes.append(new_shape)

    new_data["shapes"] = new_shapes
    new_data["imagePath"] = str(aug_image_path)
    new_data["imageData"] = None
    new_data["imageWidth"] = int(image_w)
    new_data["imageHeight"] = int(image_h)

    # 记录来源信息，方便以后排查
    new_data.setdefault("flags", {})
    new_data["flags"]["aug_type"] = aug_type
    new_data["flags"]["source_json"] = str(source_json_path)
    new_data["flags"]["source_image_augmented_to"] = str(aug_image_path)

    return new_data


def save_labelme_json(data, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def augment_no_fire_offline(image, aug_type):
    if aug_type == "hflip":
        return ImageOps.mirror(image)

    if aug_type == "vflip":
        return ImageOps.flip(image)

    if aug_type == "hvflip":
        return ImageOps.flip(ImageOps.mirror(image))

    if aug_type == "rotate_pos":
        return image.rotate(
            8,
            resample=RESAMPLE,
            expand=False,
            fillcolor=(127, 127, 127)
        )

    if aug_type == "rotate_neg":
        return image.rotate(
            -8,
            resample=RESAMPLE,
            expand=False,
            fillcolor=(127, 127, 127)
        )

    raise ValueError(f"未知 no_fire 增强类型: {aug_type}")


def augment_fire_offline(image, aug_type):
    """
    fire 增强建议只做 hflip / vflip / hvflip。
    不建议对 fire bbox 做旋转增强，因为矩形框旋转后需要更复杂的框变换，
    而且旋转会引入边缘填充，对小火样本不稳定。
    """
    if aug_type == "hflip":
        return ImageOps.mirror(image)

    if aug_type == "vflip":
        return ImageOps.flip(image)

    if aug_type == "hvflip":
        return ImageOps.flip(ImageOps.mirror(image))

    raise ValueError(f"未知 fire 增强类型: {aug_type}")


# =============
# 读取原始标签，检查图片有效性，并挂接 fire 标注 json
# =============

if not JSON_PATH.exists():
    raise FileNotFoundError(f"未找到标签文件: {JSON_PATH}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"未找到图片目录: {IMAGE_DIR}")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    label_dict = json.load(f)

records = []

for filename, label in label_dict.items():
    image_path = IMAGE_DIR / filename

    item = {
        "filename": filename,
        "path": str(image_path),
        "label": int(label),
        "exists": image_path.exists(),
        "is_valid": False,
        "width": None,
        "height": None,
        "mode": None,
        "is_augmented": False,
        "is_manual_crop": False,
        "source_filename": filename,
        "aug_type": "original",
        "source_type": "original_real_image",
        "annotation_path": "",
        "source_annotation_path": "",
        "has_fire_box": False,
        "bbox_count": 0,
        "annotation_size_match": False,
        "error": ""
    }

    if image_path.exists():
        try:
            with Image.open(image_path) as img0:
                img = img0.convert("RGB")
                w, h = img.size
                item["width"] = int(w)
                item["height"] = int(h)
                item["mode"] = "RGB"
                item["is_valid"] = True

            if int(label) == 1:
                ann_path = find_fire_box_json(filename)

                if ann_path is not None and ann_path.exists():
                    try:
                        ann_data = load_labelme_json(ann_path)
                        ann_w, ann_h = get_labelme_size(ann_data)
                        bbox_count = count_fire_shapes_from_labelme(ann_data)

                        size_match = (
                                ann_w is not None and
                                ann_h is not None and
                                int(ann_w) == int(w) and
                                int(ann_h) == int(h)
                        )

                        item["annotation_path"] = str(ann_path)
                        item["source_annotation_path"] = str(ann_path)
                        item["bbox_count"] = int(bbox_count)
                        item["annotation_size_match"] = bool(size_match)
                        item["has_fire_box"] = bool(bbox_count > 0 and size_match)

                    except Exception as e:
                        item["error"] = f"annotation_read_error: {repr(e)}"

        except Exception as e:
            item["error"] = repr(e)

    records.append(item)

raw_df = pd.DataFrame(records)

valid_df = raw_df[
    (raw_df["exists"] == True) &
    (raw_df["is_valid"] == True)
    ].copy()

valid_df["label"] = valid_df["label"].astype(int)

if REQUIRE_ANN_SIZE_MATCH:
    bad_fire_ann_df = valid_df[
        (valid_df["label"] == 1) &
        (valid_df["annotation_path"] != "") &
        (valid_df["annotation_size_match"] == False)
        ].copy()

    if len(bad_fire_ann_df) > 0:
        print("\n警告：以下 fire 图片标注尺寸与原图尺寸不一致，不能用于 fire 增强：")
        print(bad_fire_ann_df[["filename", "width", "height", "annotation_path"]].head(20))

valid_df.to_csv(
    ALL_VALID_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("\n原始有效图片数:", len(valid_df))
print("原始类别分布:")
print(valid_df["label"].value_counts().sort_index())

print("\nfire 标注情况:")
fire_df_tmp = valid_df[valid_df["label"] == 1]
print("fire 总数:", len(fire_df_tmp))
print("有可用 fire bbox 的图片数:", int(fire_df_tmp["has_fire_box"].sum()))
print("无可用 fire bbox 的图片数:", int((fire_df_tmp["has_fire_box"] == False).sum()))

print("\n有效图片统计已保存:")
print(ALL_VALID_CSV)


# =============
# 读取 manual_no_fire_crops
# =============

def load_manual_no_fire_crops(manual_dir):
    manual_dir = Path(manual_dir)

    if not manual_dir.exists():
        print("\n未检测到 manual_no_fire_crops 文件夹:", manual_dir)
        return pd.DataFrame()

    files = [
        p for p in manual_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    records = []

    for p in files:
        try:
            with Image.open(p) as img0:
                img = img0.convert("RGB")
                w, h = img.size

            records.append({
                "filename": p.name,
                "path": str(p),
                "label": 0,
                "exists": True,
                "is_valid": True,
                "width": int(w),
                "height": int(h),
                "mode": "RGB",
                "is_augmented": False,
                "is_manual_crop": True,
                "source_filename": p.name,
                "aug_type": "manual_no_fire_crop",
                "source_type": "manual_no_fire_crops",
                "annotation_path": "",
                "source_annotation_path": "",
                "has_fire_box": False,
                "bbox_count": 0,
                "annotation_size_match": False,
                "error": ""
            })

        except Exception as e:
            print("manual crop 读取失败:", p, repr(e))

    manual_df = pd.DataFrame(records)

    if len(manual_df) > 0:
        manual_df = manual_df.sample(
            frac=1,
            random_state=RANDOM_SEED
        ).reset_index(drop=True)

    print("\nmanual_no_fire_crops 数量:", len(manual_df))

    if len(manual_df) > 0:
        print("manual_no_fire_crops 尺寸统计:")
        print(manual_df[["width", "height"]].describe())

    return manual_df


manual_no_fire_df = load_manual_no_fire_crops(MANUAL_NO_FIRE_DIR)


def save_manual_no_fire_preview(manual_df, save_path, max_show=24):
    if len(manual_df) == 0:
        print("manual_no_fire_df 为空，不生成预览图。")
        return

    show_df = manual_df.head(max_show).copy()

    n = len(show_df)
    ncols = 6
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 3, nrows * 3)
    )

    if nrows == 1:
        axes = np.array([axes])

    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for i, (_, row) in enumerate(show_df.iterrows()):
        image_path = Path(row["path"])

        with Image.open(image_path) as img0:
            img = img0.convert("RGB")

        axes[i].imshow(img)
        axes[i].set_title(
            f"{row['filename']}\n{row['width']}x{row['height']}",
            fontsize=8
        )
        axes[i].axis("off")

    fig.suptitle(
        "manual_no_fire_crops preview: all treated as label=0 train augmentation",
        fontsize=14
    )

    fig.tight_layout()
    fig.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )
    plt.close(fig)

    print("\nmanual_no_fire_crops 示例图已保存:")
    print(save_path)


manual_preview_path = TRAIN_META_DIR / "manual_no_fire_crops_preview.png"

save_manual_no_fire_preview(
    manual_df=manual_no_fire_df,
    save_path=manual_preview_path,
    max_show=MAX_MANUAL_PREVIEW
)


# =============
# 离线增强：no_fire 只保存图片；fire 同时保存图片和变换后的 bbox json
# =============

def create_offline_aug_records(
        source_df,
        target_count,
        save_dir,
        label,
        class_name,
        aug_types,
        fire_json_save_dir=None
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    if fire_json_save_dir is not None:
        fire_json_save_dir = Path(fire_json_save_dir)
        fire_json_save_dir.mkdir(parents=True, exist_ok=True)

    if target_count <= 0:
        return pd.DataFrame()

    if len(source_df) == 0:
        raise ValueError(f"{class_name} source_df 为空，无法离线增强。")

    if int(label) == 1 and REQUIRE_FIRE_BOX_FOR_FIRE_AUG:
        source_df = source_df[
            (source_df["has_fire_box"] == True) &
            (source_df["annotation_path"].astype(str) != "")
            ].copy()

        if len(source_df) == 0:
            raise ValueError(
                "需要生成 fire 增强图，但 train_real_fire_df 中没有可用 fire bbox 标注。\n"
                "请检查 FIRE_BOX_JSON_DIR 是否正确，或者检查 LabelMe json 的 imageWidth/imageHeight 是否与原图一致。"
            )

    records = []

    for i in range(target_count):
        row = source_df.iloc[i % len(source_df)]

        src_path = Path(str(row["path"]))
        if not src_path.exists():
            src_path = IMAGE_DIR / str(row["filename"])

        aug_type = aug_types[i % len(aug_types)]

        with Image.open(src_path) as img0:
            image = img0.convert("RGB")

        if int(label) == 0:
            aug_image = augment_no_fire_offline(image, aug_type)

        else:
            aug_image = augment_fire_offline(image, aug_type)

        aug_filename = (
            f"aug_{class_name}_{i:05d}_"
            f"{aug_type}_{Path(str(row['filename'])).stem}.jpg"
        )

        aug_path = save_dir / aug_filename
        aug_image.save(aug_path, quality=95)

        annotation_path = ""
        source_annotation_path = ""
        has_fire_box = False
        bbox_count = 0
        annotation_size_match = False

        if int(label) == 1:
            source_annotation_path = str(row["annotation_path"])

            if source_annotation_path == "" or not Path(source_annotation_path).exists():
                raise FileNotFoundError(
                    f"fire 增强需要原始标注 json，但未找到: {source_annotation_path}"
                )

            aug_json_filename = Path(aug_filename).with_suffix(".json").name
            aug_json_path = fire_json_save_dir / aug_json_filename

            aug_labelme = transform_labelme_for_aug(
                source_json_path=source_annotation_path,
                aug_type=aug_type,
                image_w=aug_image.size[0],
                image_h=aug_image.size[1],
                aug_image_filename=aug_filename,
                aug_image_path=aug_path
            )

            save_labelme_json(
                data=aug_labelme,
                save_path=aug_json_path
            )

            annotation_path = str(aug_json_path)
            has_fire_box = True
            bbox_count = count_fire_shapes_from_labelme(aug_labelme)
            annotation_size_match = True

        records.append({
            "filename": aug_filename,
            "path": str(aug_path),
            "label": int(label),
            "exists": True,
            "is_valid": True,
            "width": int(aug_image.size[0]),
            "height": int(aug_image.size[1]),
            "mode": "RGB",
            "is_augmented": True,
            "is_manual_crop": False,
            "source_filename": str(row["filename"]),
            "aug_type": aug_type,
            "source_type": f"offline_aug_from_{class_name}",
            "annotation_path": annotation_path,
            "source_annotation_path": source_annotation_path,
            "has_fire_box": bool(has_fire_box),
            "bbox_count": int(bbox_count),
            "annotation_size_match": bool(annotation_size_match),
            "error": ""
        })

    return pd.DataFrame(records)


# =============
# 正式划分：val/test 只用真实原图，train 使用真实图 + manual no_fire + 离线增强
# =============

if FORCE_RESPLIT:
    if ENSEMBLE_DIR.exists():
        shutil.rmtree(ENSEMBLE_DIR)
    ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

    if AUG_IMAGE_DIR.exists():
        shutil.rmtree(AUG_IMAGE_DIR)

    if AUG_BOX_JSON_DIR.exists():
        shutil.rmtree(AUG_BOX_JSON_DIR)

    AUG_NO_FIRE_DIR.mkdir(parents=True, exist_ok=True)
    AUG_FIRE_DIR.mkdir(parents=True, exist_ok=True)
    AUG_FIRE_BOX_JSON_DIR.mkdir(parents=True, exist_ok=True)


def mark_original_rows(df):
    out = df.copy()
    out["is_augmented"] = False
    out["is_manual_crop"] = False
    out["source_filename"] = out["filename"]
    out["aug_type"] = "original"
    out["source_type"] = "original_real_image"
    return out


df0 = valid_df[valid_df["label"] == 0].sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

df1 = valid_df[valid_df["label"] == 1].sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

if len(df0) < TEST_PER_CLASS + VAL_PER_CLASS:
    raise ValueError(
        f"\n真实 no_fire 原图不足。\n"
        f"当前真实 no_fire = {len(df0)}\n"
        f"至少需要 TEST_PER_CLASS + VAL_PER_CLASS = {TEST_PER_CLASS + VAL_PER_CLASS}\n"
        f"注意：manual_no_fire_crops 只能进入 train，不能用于 val/test。"
    )

if len(df1) < TEST_PER_CLASS + VAL_PER_CLASS:
    raise ValueError(
        f"\n真实 fire 原图不足。\n"
        f"当前真实 fire = {len(df1)}\n"
        f"至少需要 TEST_PER_CLASS + VAL_PER_CLASS = {TEST_PER_CLASS + VAL_PER_CLASS}"
    )

test_no_fire_df = mark_original_rows(
    df0.iloc[:TEST_PER_CLASS].reset_index(drop=True)
)

test_fire_df = mark_original_rows(
    df1.iloc[:TEST_PER_CLASS].reset_index(drop=True)
)

val_no_fire_df = mark_original_rows(
    df0.iloc[
        TEST_PER_CLASS:TEST_PER_CLASS + VAL_PER_CLASS
    ].reset_index(drop=True)
)

val_fire_df = mark_original_rows(
    df1.iloc[
        TEST_PER_CLASS:TEST_PER_CLASS + VAL_PER_CLASS
    ].reset_index(drop=True)
)

train_real_no_fire_df = mark_original_rows(
    df0.iloc[
        TEST_PER_CLASS + VAL_PER_CLASS:
    ].reset_index(drop=True)
)

train_real_fire_df = mark_original_rows(
    df1.iloc[
        TEST_PER_CLASS + VAL_PER_CLASS:
    ].reset_index(drop=True)
)

print("\n真实原图划分:")
print("test no_fire:", len(test_no_fire_df), "test fire:", len(test_fire_df))
print("val  no_fire:", len(val_no_fire_df), "val  fire:", len(val_fire_df))
print("train real no_fire:", len(train_real_no_fire_df))
print("train real fire:", len(train_real_fire_df))
print("train real fire 有可用 bbox:", int(train_real_fire_df["has_fire_box"].sum()))
print("manual no_fire crops:", len(manual_no_fire_df))

# =============
# 构建 no_fire 训练池
# =============

if USE_MANUAL_NO_FIRE_CROPS and len(manual_no_fire_df) > 0:
    no_fire_train_pool_df = pd.concat(
        [train_real_no_fire_df, manual_no_fire_df],
        axis=0,
        ignore_index=True
    ).sample(
        frac=1,
        random_state=RANDOM_SEED
    ).reset_index(drop=True)

    print("\n已将 manual_no_fire_crops 加入 no_fire 训练池")
    print("真实 no_fire:", len(train_real_no_fire_df))
    print("手动裁剪 no_fire:", len(manual_no_fire_df))
    print("no_fire 训练池总数:", len(no_fire_train_pool_df))

else:
    no_fire_train_pool_df = train_real_no_fire_df.copy().reset_index(drop=True)

    print("\n未使用 manual_no_fire_crops")
    print("真实 no_fire:", len(train_real_no_fire_df))
    print("no_fire 训练池总数:", len(no_fire_train_pool_df))

# 原逻辑只在 no_fire 数量不足 TRAIN_PER_CLASS 时才增强。
# 现在额外加入 NO_FIRE_FORCE_EXTRA_AUG_COUNT，避免训练池够数时增强文件夹为空。
need_no_fire_fill_count = max(0, TRAIN_PER_CLASS - len(no_fire_train_pool_df))
no_fire_extra_aug_count = int(NO_FIRE_FORCE_EXTRA_AUG_COUNT) if USE_NO_FIRE_OFFLINE_AUG else 0
need_no_fire_aug = need_no_fire_fill_count + no_fire_extra_aug_count

print("\nno_fire 增强需求:")
print("need_no_fire_fill_count:", need_no_fire_fill_count)
print("no_fire_extra_aug_count:", no_fire_extra_aug_count)
print("need_no_fire_aug total:", need_no_fire_aug)

if USE_NO_FIRE_OFFLINE_AUG and need_no_fire_aug > 0:
    no_fire_aug_df = create_offline_aug_records(
        source_df=no_fire_train_pool_df,
        target_count=need_no_fire_aug,
        save_dir=AUG_NO_FIRE_DIR,
        label=0,
        class_name="no_fire",
        aug_types=NO_FIRE_AUG_TYPES,
        fire_json_save_dir=None
    )

    no_fire_train_pool_df = pd.concat(
        [no_fire_train_pool_df, no_fire_aug_df],
        axis=0,
        ignore_index=True
    ).sample(
        frac=1,
        random_state=RANDOM_SEED
    ).reset_index(drop=True)

else:
    no_fire_aug_df = pd.DataFrame()

print("\nno_fire 训练池补齐后总数:", len(no_fire_train_pool_df))
print("no_fire 离线增强新增:", len(no_fire_aug_df))

# =============
# 构建 fire 训练池，并生成 fire 增强图对应的 bbox json
# =============

fire_aug_fill_count = max(0, TRAIN_PER_CLASS - len(train_real_fire_df))

fire_annotated_source_count = int(
    (
            (train_real_fire_df["has_fire_box"] == True) &
            (train_real_fire_df["annotation_path"].astype(str) != "")
    ).sum()
)

fire_extra_aug_count = fire_annotated_source_count * int(FIRE_EXTRA_AUG_PER_ORIGINAL)

need_fire_aug = fire_aug_fill_count + fire_extra_aug_count

if USE_FIRE_OFFLINE_AUG and need_fire_aug > 0:
    fire_aug_source_df = train_real_fire_df.copy()

    if REQUIRE_FIRE_BOX_FOR_FIRE_AUG:
        fire_aug_source_df = fire_aug_source_df[
            (fire_aug_source_df["has_fire_box"] == True) &
            (fire_aug_source_df["annotation_path"].astype(str) != "")
            ].reset_index(drop=True)

    fire_aug_df = create_offline_aug_records(
        source_df=fire_aug_source_df,
        target_count=need_fire_aug,
        save_dir=AUG_FIRE_DIR,
        label=1,
        class_name="fire",
        aug_types=FIRE_AUG_TYPES,
        fire_json_save_dir=AUG_FIRE_BOX_JSON_DIR
    )

    fire_train_pool_df = pd.concat(
        [train_real_fire_df, fire_aug_df],
        axis=0,
        ignore_index=True
    ).sample(
        frac=1,
        random_state=RANDOM_SEED
    ).reset_index(drop=True)

else:
    fire_aug_df = pd.DataFrame()
    fire_train_pool_df = train_real_fire_df.copy().reset_index(drop=True)

print("\nfire 训练池总数:", len(fire_train_pool_df))
print("fire 离线增强新增:", len(fire_aug_df))
print("fire 增强标注 json 保存目录:", AUG_FIRE_BOX_JSON_DIR)

if len(fire_aug_df) > 0:
    print("\nfire 增强样例:")
    print(fire_aug_df[[
        "filename",
        "path",
        "annotation_path",
        "source_filename",
        "source_annotation_path",
        "aug_type",
        "bbox_count"
    ]].head())

# =============
# 保存固定 val/test 与 train_base_pool
# =============

test_df = pd.concat(
    [test_no_fire_df, test_fire_df],
    axis=0,
    ignore_index=True
).sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

val_df = pd.concat(
    [val_no_fire_df, val_fire_df],
    axis=0,
    ignore_index=True
).sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

train_base_df = pd.concat(
    [no_fire_train_pool_df, fire_train_pool_df],
    axis=0,
    ignore_index=True
).sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

test_df.to_csv(
    TEST_CSV,
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    VAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

train_base_df.to_csv(
    TRAIN_BASE_CSV,
    index=False,
    encoding="utf-8-sig"
)

# =============
# 生成 ensemble_train_xx.csv
# =============

ensemble_summary = []

for ensemble_id in range(N_ENSEMBLE):
    seed_i = RANDOM_SEED + 1000 + ensemble_id

    replace_no_fire = len(no_fire_train_pool_df) < TRAIN_PER_CLASS
    replace_fire = len(fire_train_pool_df) < TRAIN_PER_CLASS

    sampled_no_fire_df = no_fire_train_pool_df.sample(
        n=TRAIN_PER_CLASS,
        replace=replace_no_fire,
        random_state=seed_i
    )

    sampled_fire_df = fire_train_pool_df.sample(
        n=TRAIN_PER_CLASS,
        replace=replace_fire,
        random_state=seed_i
    )

    ensemble_train_df = pd.concat(
        [sampled_no_fire_df, sampled_fire_df],
        axis=0,
        ignore_index=True
    ).sample(
        frac=1,
        random_state=seed_i
    ).reset_index(drop=True)

    save_path = ENSEMBLE_DIR / f"ensemble_train_{ensemble_id:02d}.csv"

    ensemble_train_df.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    label_counts = ensemble_train_df["label"].value_counts().sort_index()

    summary_row = {
        "ensemble_id": ensemble_id,
        "csv_path": str(save_path),
        "total": len(ensemble_train_df),
        "no_fire_0": int(label_counts.get(0, 0)),
        "fire_1": int(label_counts.get(1, 0)),

        "real_no_fire": int(
            (
                    (ensemble_train_df["label"] == 0) &
                    (ensemble_train_df["is_augmented"] == False) &
                    (ensemble_train_df["is_manual_crop"] == False)
            ).sum()
        ),
        "manual_no_fire_crop": int(
            (
                    (ensemble_train_df["label"] == 0) &
                    (ensemble_train_df["is_manual_crop"] == True)
            ).sum()
        ),
        "offline_aug_no_fire": int(
            (
                    (ensemble_train_df["label"] == 0) &
                    (ensemble_train_df["is_augmented"] == True)
            ).sum()
        ),

        "real_fire": int(
            (
                    (ensemble_train_df["label"] == 1) &
                    (ensemble_train_df["is_augmented"] == False)
            ).sum()
        ),
        "offline_aug_fire": int(
            (
                    (ensemble_train_df["label"] == 1) &
                    (ensemble_train_df["is_augmented"] == True)
            ).sum()
        ),

        "fire_with_bbox": int(
            (
                    (ensemble_train_df["label"] == 1) &
                    (ensemble_train_df["has_fire_box"] == True)
            ).sum()
        ),
        "fire_without_bbox": int(
            (
                    (ensemble_train_df["label"] == 1) &
                    (ensemble_train_df["has_fire_box"] == False)
            ).sum()
        ),
        "aug_fire_with_generated_bbox_json": int(
            (
                    (ensemble_train_df["label"] == 1) &
                    (ensemble_train_df["is_augmented"] == True) &
                    (ensemble_train_df["annotation_path"].astype(str) != "")
            ).sum()
        )
    }

    ensemble_summary.append(summary_row)

ensemble_summary_df = pd.DataFrame(ensemble_summary)

ensemble_summary_df.to_csv(
    ENSEMBLE_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# =============
# 保存构成统计
# =============

def composition_summary(df, name):
    rows = []

    rows.append({
        "dataset": name,
        "item": "total",
        "count": len(df)
    })

    for label, cnt in df["label"].value_counts().sort_index().items():
        rows.append({
            "dataset": name,
            "item": f"label_{label}",
            "count": int(cnt)
        })

    if "source_type" in df.columns:
        for source_type, cnt in df["source_type"].value_counts().items():
            rows.append({
                "dataset": name,
                "item": f"source_type::{source_type}",
                "count": int(cnt)
            })

    if "aug_type" in df.columns:
        for aug_type, cnt in df["aug_type"].value_counts().items():
            rows.append({
                "dataset": name,
                "item": f"aug_type::{aug_type}",
                "count": int(cnt)
            })

    if "has_fire_box" in df.columns:
        fire_df = df[df["label"] == 1]
        rows.append({
            "dataset": name,
            "item": "fire_has_bbox",
            "count": int((fire_df["has_fire_box"] == True).sum())
        })
        rows.append({
            "dataset": name,
            "item": "fire_no_bbox",
            "count": int((fire_df["has_fire_box"] == False).sum())
        })

    return rows


summary_rows = []
summary_rows += composition_summary(test_df, "test")
summary_rows += composition_summary(val_df, "val")
summary_rows += composition_summary(train_base_df, "train_base_pool")

for ensemble_id in range(N_ENSEMBLE):
    ensemble_train_df = pd.read_csv(
        ENSEMBLE_DIR / f"ensemble_train_{ensemble_id:02d}.csv"
    )

    summary_rows += composition_summary(
        ensemble_train_df,
        f"ensemble_train_{ensemble_id:02d}"
    )

composition_df = pd.DataFrame(summary_rows)

composition_summary_csv = TRAIN_META_DIR / "train_composition_summary.csv"

composition_df.to_csv(
    composition_summary_csv,
    index=False,
    encoding="utf-8-sig"
)


# =============
# 强制检查：训练 attention 所需 metadata 必须存在
# =============

REQUIRED_ATTENTION_COLUMNS = [
    "annotation_path",
    "source_annotation_path",
    "has_fire_box",
    "bbox_count",
    "annotation_size_match"
]

def assert_attention_columns_exist(df, name):
    missing_cols = [c for c in REQUIRED_ATTENTION_COLUMNS if c not in df.columns]
    if missing_cols:
        raise RuntimeError(
            f"{name} 缺少 attention/bbox 必要列: {missing_cols}。\n"
            "这会导致训练时 bboxes=[]，attention loss 永远为 0。"
        )

    fire_df = df[df["label"] == 1].copy()
    fire_with_box = int((fire_df["has_fire_box"] == True).sum())
    print(f"{name} fire_with_box: {fire_with_box}/{len(fire_df)}")

    return fire_with_box

assert_attention_columns_exist(test_df, "test_df")
assert_attention_columns_exist(val_df, "val_df")
assert_attention_columns_exist(train_base_df, "train_base_df")

for ensemble_id in range(N_ENSEMBLE):
    _tmp_df = pd.read_csv(ENSEMBLE_DIR / f"ensemble_train_{ensemble_id:02d}.csv")
    assert_attention_columns_exist(_tmp_df, f"ensemble_train_{ensemble_id:02d}")


# =============
# 输出检查
# =============

print("\n划分完成。")

print("\nTEST 分布，应全部是真实原图:")
print(test_df["label"].value_counts().sort_index())
print(test_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\nVAL 分布，应全部是真实原图:")
print(val_df["label"].value_counts().sort_index())
print(val_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\nTRAIN_BASE_POOL 分布:")
print(train_base_df["label"].value_counts().sort_index())

print("\nTRAIN_BASE_POOL source_type:")
print(train_base_df["source_type"].value_counts())

print("\nTRAIN_BASE_POOL fire bbox 情况:")
fire_train_base = train_base_df[train_base_df["label"] == 1]
print("fire total:", len(fire_train_base))
print("fire has bbox:", int((fire_train_base["has_fire_box"] == True).sum()))
print("fire no bbox:", int((fire_train_base["has_fire_box"] == False).sum()))
print("aug fire with generated json:", int(
    (
            (fire_train_base["is_augmented"] == True) &
            (fire_train_base["annotation_path"].astype(str) != "")
    ).sum()
))

print("\nENSEMBLE SUMMARY:")
print(ensemble_summary_df)

print("\n构成统计保存到:")
print(composition_summary_csv)

print("\n重要输出文件:")
print("TEST_CSV:", TEST_CSV)
print("VAL_CSV:", VAL_CSV)
print("TRAIN_BASE_CSV:", TRAIN_BASE_CSV)
print("ENSEMBLE_SUMMARY_CSV:", ENSEMBLE_SUMMARY_CSV)
print("ENSEMBLE_DIR:", ENSEMBLE_DIR)
print("AUG_FIRE_DIR:", AUG_FIRE_DIR)
print("AUG_FIRE_BOX_JSON_DIR:", AUG_FIRE_BOX_JSON_DIR)

# =============
# 读取划分结果并检查
# =============

test_df = pd.read_csv(TEST_CSV)
val_df = pd.read_csv(VAL_CSV)
train_base_df = pd.read_csv(TRAIN_BASE_CSV)
ensemble_summary_df = pd.read_csv(ENSEMBLE_SUMMARY_CSV)
example_train_df = pd.read_csv(ENSEMBLE_DIR / "ensemble_train_00.csv")

print("\n测试集分布:")
print(test_df["label"].value_counts().sort_index())

print("\n验证集分布:")
print(val_df["label"].value_counts().sort_index())

print("\n基础训练池分布:")
print(train_base_df["label"].value_counts().sort_index())

print("\n测试集增强/裁切检查，应全部为 False:")
print(test_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\n验证集增强/裁切检查，应全部为 False:")
print(val_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\n集成训练子集:")
print(ensemble_summary_df)

print("\nensemble_train_00 类别分布:")
print(example_train_df["label"].value_counts().sort_index())

print("\nensemble_train_00 no_fire 来源统计:")
print(
    example_train_df[
        example_train_df["label"] == 0
        ][["is_augmented", "is_manual_crop", "aug_type"]].value_counts()
)

print("\nensemble_train_00 fire 来源与 bbox 统计:")
print(
    example_train_df[
        example_train_df["label"] == 1
        ][["is_augmented", "has_fire_box", "aug_type"]].value_counts()
)

print("\nensemble_train_00 增强 fire 标注样例:")
print(
    example_train_df[
        (example_train_df["label"] == 1) &
        (example_train_df["is_augmented"] == True)
        ][[
        "filename",
        "path",
        "annotation_path",
        "source_filename",
        "source_annotation_path",
        "aug_type",
        "bbox_count"
    ]].head()
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

test_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Fixed test, real only")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["0", "1"], rotation=0)

val_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Fixed validation, real only")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels(["0", "1"], rotation=0)

example_train_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[2])
axes[2].set_title("Ensemble train 00")
axes[2].set_xlabel("Label")
axes[2].set_ylabel("Count")
axes[2].set_xticklabels(["0", "1"], rotation=0)

fig.tight_layout()
fig.savefig(SPLIT_DIR / "split_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("\n划分统计图已保存:")
print(SPLIT_DIR / "split_distribution.png")

ROOT: E:\Programming\Python\DeepLearning\比赛
JSON_PATH: E:\Programming\Python\DeepLearning\比赛\train_image.json
IMAGE_DIR: E:\Programming\Python\DeepLearning\比赛\images
FIRE_BOX_JSON_DIR: E:\Programming\Python\DeepLearning\比赛\images_fire
SPLIT_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits
AUG_FIRE_BOX_JSON_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits\augmented_box_jsons\fire

警告：以下 fire 图片标注尺寸与原图尺寸不一致，不能用于 fire 增强：
                         filename  width  height  \
489  20250526_firesmoke_00845.jpg    720    1280   

                                       annotation_path  
489  E:\Programming\Python\DeepLearning\比赛\images_f...  

原始有效图片数: 1100
原始类别分布:
label
0    236
1    864
Name: count, dtype: int64

fire 标注情况:
fire 总数: 864
有可用 fire bbox 的图片数: 863
无可用 fire bbox 的图片数: 1

有效图片统计已保存:
E:\Programming\Python\DeepLearning\比赛\fire_splits\all_valid_images.csv

manual_no_fire_crops 数量: 537
manual_no_fire_crops 尺寸统计:
             width       height
count   537.000000   537.00000

In [3]:
# =============
# 统计划分后所有数据集大小
# =============

dataset_csvs = [
    ("test", TEST_CSV),
    ("val", VAL_CSV),
    ("train_base_pool", TRAIN_BASE_CSV),
]

# 统计所有 ensemble_train_xx.csv，不包含 _hard_weighted.csv
for p in sorted(ENSEMBLE_DIR.glob("ensemble_train_*.csv")):
    if not p.name.endswith("_hard_weighted.csv"):
        dataset_csvs.append((p.stem, p))

summary_rows = []
all_paths = []

for dataset_name, csv_path in dataset_csvs:
    if not csv_path.exists():
        print(f"跳过，不存在: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df["label"] = df["label"].astype(int)

    total = len(df)
    no_fire = int((df["label"] == 0).sum())
    fire = int((df["label"] == 1).sum())

    manual_crop = int(df["is_manual_crop"].sum()) if "is_manual_crop" in df.columns else 0
    augmented = int(df["is_augmented"].sum()) if "is_augmented" in df.columns else 0

    real_original = total - manual_crop - augmented

    if "path" in df.columns:
        all_paths.extend(df["path"].dropna().astype(str).tolist())

    summary_rows.append({
        "dataset": dataset_name,
        "csv_path": str(csv_path),
        "total": total,
        "no_fire_0": no_fire,
        "fire_1": fire,
        "real_original": real_original,
        "manual_no_fire_crop": manual_crop,
        "offline_augmented": augmented,
    })

summary_df = pd.DataFrame(summary_rows)

print("划分后各数据集大小统计：")
display(summary_df)

print("\n按训练任务累计样本数：")
print("total rows across listed csvs:", int(summary_df["total"].sum()))
print("total no_fire rows:", int(summary_df["no_fire_0"].sum()))
print("total fire rows:", int(summary_df["fire_1"].sum()))

print("\n去重后的图片路径数量：")
print("unique image paths:", len(set(all_paths)))

save_path = TRAIN_META_DIR / "split_dataset_size_summary.csv"
summary_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("\n统计结果已保存：")
print(save_path)

划分后各数据集大小统计：


,dataset,csv_path,total,no_fire_0,fire_1,real_original,manual_no_fire_crop,offline_augmented
0,test,E:\Programming\Python\DeepLearning\比赛\fire_spl...,300,150,150,300,0,0
1,val,E:\Programming\Python\DeepLearning\比赛\fire_spl...,100,50,50,100,0,0
2,train_base_pool,E:\Programming\Python\DeepLearning\比赛\fire_spl...,2100,773,1327,700,537,863
3,ensemble_train_00,E:\Programming\Python\DeepLearning\比赛\fire_spl...,1000,500,500,277,348,375



按训练任务累计样本数：
total rows across listed csvs: 3500
total no_fire rows: 1473
total fire rows: 2027

去重后的图片路径数量：
unique image paths: 2500

统计结果已保存：
E:\Programming\Python\DeepLearning\比赛\fire_splits\train\split_dataset_size_summary.csv


In [4]:
# =============
# 读取划分结果并检查
# =============

test_df = pd.read_csv(TEST_CSV)
val_df = pd.read_csv(VAL_CSV)
train_base_df = pd.read_csv(TRAIN_BASE_CSV)
ensemble_summary_df = pd.read_csv(ENSEMBLE_SUMMARY_CSV)
example_train_df = pd.read_csv(ENSEMBLE_DIR / "ensemble_train_00.csv")

print("\n测试集分布:")
print(test_df["label"].value_counts().sort_index())

print("\n验证集分布:")
print(val_df["label"].value_counts().sort_index())

print("\n基础训练池分布:")
print(train_base_df["label"].value_counts().sort_index())

print("\n测试集增强/裁切检查，应全部为 False:")
print(test_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\n验证集增强/裁切检查，应全部为 False:")
print(val_df[["is_augmented", "is_manual_crop"]].value_counts())

print("\n集成训练子集:")
print(ensemble_summary_df)

print("\nensemble_train_00 类别分布:")
print(example_train_df["label"].value_counts().sort_index())

print("\nensemble_train_00 no_fire 来源统计:")
print(example_train_df[example_train_df["label"] == 0][["is_augmented", "is_manual_crop", "aug_type"]].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

test_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Fixed test, real only")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["0", "1"], rotation=0)

val_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Fixed validation, real only")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels(["0", "1"], rotation=0)

example_train_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[2])
axes[2].set_title("Ensemble train 00")
axes[2].set_xlabel("Label")
axes[2].set_ylabel("Count")
axes[2].set_xticklabels(["0", "1"], rotation=0)

fig.tight_layout()
fig.savefig(SPLIT_DIR / "split_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("\n划分统计图已保存:")
print(SPLIT_DIR / "split_distribution.png")


测试集分布:
label
0    150
1    150
Name: count, dtype: int64

验证集分布:
label
0    50
1    50
Name: count, dtype: int64

基础训练池分布:
label
0     773
1    1327
Name: count, dtype: int64

测试集增强/裁切检查，应全部为 False:
is_augmented  is_manual_crop
False         False             300
Name: count, dtype: int64

验证集增强/裁切检查，应全部为 False:
is_augmented  is_manual_crop
False         False             100
Name: count, dtype: int64

集成训练子集:
   ensemble_id                                           csv_path  total  \
0            0  E:\Programming\Python\DeepLearning\比赛\fire_spl...   1000   

   no_fire_0  fire_1  real_no_fire  manual_no_fire_crop  offline_aug_no_fire  \
0        500     500            24                  348                  128   

   real_fire  offline_aug_fire  fire_with_bbox  fire_without_bbox  \
0        253               247             499                  1   

   aug_fire_with_generated_bbox_json  
0                                247  

ensemble_train_00 类别分布:
label
0    500
1    500
Name: